# Setup and Validation Notebook

Notebook ini untuk:
1. Install dependencies di Colab
2. Upload dataset dari lokal ke Colab runtime
3. Check GPU availability
4. Load dan validate datasets
5. Test tokenizer
6. Setup model dengan LoRA
7. Baseline evaluation

> **Catatan:** Notebook ini dijalankan via VS Code yang terhubung ke Colab kernel.
> Dataset di-upload dari lokal ke Colab runtime (`/content/`).

## 1. Install Dependencies

In [1]:
# Install semua dependencies yang dibutuhkan di Colab runtime
# (berbeda dengan conda env lokal - harus install ulang)
!pip install -q transformers peft datasets accelerate
!pip install -q evaluate rouge_score bert_score
print('✓ Dependencies installed')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.2 MB/s eta 0:00:00
✓ Dependencies installed


In [2]:
# Cek versi semua library yang terinstall
import importlib
import subprocess
import sys, torch, platform

libs = [
    "transformers",
    "peft", 
    "datasets",
    "accelerate",
    "evaluate",
    "torch",
    "tokenizers",
    "rouge_score",
    "bert_score",
]


print(f"Python:  {sys.version}")
print(f"OS:      {platform.system()}")
print(f"Torch:   {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
print(f"\n ")

print("=== Library Versions ===")
for lib in libs:
    try:
        mod = importlib.import_module(lib.replace("-", "_"))
        version = getattr(mod, "__version__", "unknown")
        print(f"  {lib:<20} {version}")
    except ImportError:
        print(f"  {lib:<20} NOT INSTALLED")

# Cek Python version
import sys
print(f"\n  {'python':<20} {sys.version.split()[0]}")

# Cek CUDA
import torch
print(f"  {'cuda available':<20} {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {'cuda version':<20} {torch.version.cuda}")
    print(f"  {'gpu name':<20} {torch.cuda.get_device_name(0)}")


Python:  3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OS:      Linux
Torch:   2.10.0+cu128
CUDA:    True

 
=== Library Versions ===
  transformers         5.0.0
  peft                 0.18.1
  datasets             4.0.0
  accelerate           1.13.0
  evaluate             0.4.6
  torch                2.10.0+cu128
  tokenizers           0.22.2
  rouge_score          unknown
  bert_score           0.3.12

  python               3.12.13
  cuda available       True
  cuda version         12.8
  gpu name             Tesla T4


In [3]:
import tensorflow as tf
tf.test.gpu_device_name()
# Output should indicate a GPU (e.g., '/device:GPU:0')

!nvidia-smi



Fri Apr 10 15:28:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P0             27W /   70W |     105MiB /  15360MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload Dataset ke Colab Runtime

Semua file diakses via Google Drive yang sudah di-mount di section sebelumnya.

Struktur folder yang dibutuhkan di Google Drive (`MyDrive/dataset-aqg/`):
```
dataset-aqg/
├── output_domain/
│   ├── train.jsonl
│   ├── validation.jsonl
│   └── test.jsonl
├── dataset-task-spesifc/
│   ├── train.jsonl
│   ├── validation.jsonl
│   └── test.jsonl
└── src_finetuned.zip
```

In [4]:
from google.colab import drive
import shutil, os

# Mount Google Drive
drive.mount('/content/drive')

# Path file di Google Drive kamu
drive_path = '/content/drive/MyDrive/dataset_aqg/output_domain'

# Pastikan direktori tujuan ada
os.makedirs('/content/dataset_aqg/output_domain', exist_ok=True)

# Salin file ke direktori Colab
files_to_copy = ['train.jsonl', 'validation.jsonl', 'test.jsonl']
for f in files_to_copy:
    src = f'{drive_path}/{f}'
    dst = f'/content/dataset_aqg/output_domain/{f}'
    shutil.copy(src, dst)
    print(f'✓ {f} → /content/dataset_aqg/output_domain/')

print('\nSemua file berhasil disalin!')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ train.jsonl → /content/dataset_aqg/output_domain/
✓ validation.jsonl → /content/dataset_aqg/output_domain/
✓ test.jsonl → /content/dataset_aqg/output_domain/

Semua file berhasil disalin!


In [5]:
# Upload file dataset task-specific via Google Drive
# Pastikan folder 'dataset-task-spesifc' sudah ada di Google Drive kamu
# Path Drive: MyDrive/dataset-aqg/dataset-task-spesifc/

import shutil, os

drive_path_task = '/content/drive/MyDrive/dataset_aqg/dataset-task-spesifc'

os.makedirs('/content/dataset_aqg/dataset-task-spesifc', exist_ok=True)

files_to_copy = ['train.jsonl', 'validation.jsonl', 'test.jsonl']
for f in files_to_copy:
    src = f'{drive_path_task}/{f}'
    dst = f'/content/dataset_aqg/dataset-task-spesifc/{f}'
    shutil.copy(src, dst)
    print(f'✓ {f} → /content/dataset_aqg/dataset-task-spesifc/')

print('\nSemua file task-specific berhasil disalin!')

✓ train.jsonl → /content/dataset_aqg/dataset-task-spesifc/
✓ validation.jsonl → /content/dataset_aqg/dataset-task-spesifc/
✓ test.jsonl → /content/dataset_aqg/dataset-task-spesifc/

Semua file task-specific berhasil disalin!


In [6]:
# Upload source code (src/finetuned/) via Google Drive
# Pastikan src_finetuned.zip sudah ada di Google Drive kamu
# Path Drive: MyDrive/dataset-aqg/src_finetuned.zip
#
# Buat zip dulu di lokal (jalankan sekali di terminal lokal):
#   cd D:\2-Project\AQG
#   python -m zipfile -c src_finetuned.zip src/finetuned
# Lalu upload src_finetuned.zip ke Google Drive folder dataset-aqg/

import zipfile, shutil, os

zip_src = '/content/drive/MyDrive/dataset_aqg/src_finetuned.zip'
zip_dst = '/content/src_finetuned.zip'

# Salin zip dari Drive ke Colab runtime
shutil.copy(zip_src, zip_dst)
print(f'✓ src_finetuned.zip disalin dari Drive')

# Extract
with zipfile.ZipFile(zip_dst, 'r') as z:
    z.extractall('/content/')
print('✓ src_finetuned.zip extracted ke /content/')

✓ src_finetuned.zip disalin dari Drive
✓ src_finetuned.zip extracted ke /content/


In [7]:
# Setup Python path agar module bisa di-import
import sys, os
sys.path.insert(0, '/content')
os.chdir('/content')

print('Python path:', sys.path[:3])
print('Working dir:', os.getcwd())
print()

# Verifikasi semua file dataset tersedia
checks = [
    '/content/dataset_aqg/output_domain/train.jsonl',
    '/content/dataset_aqg/output_domain/validation.jsonl',
    '/content/dataset_aqg/output_domain/test.jsonl',
    '/content/dataset_aqg/dataset-task-spesifc/train.jsonl',
    '/content/dataset_aqg/dataset-task-spesifc/validation.jsonl',
    '/content/dataset_aqg/dataset-task-spesifc/test.jsonl',
]
print('=== File Check ===')
all_ok = True
for path in checks:
    exists = os.path.exists(path)
    status = '✓' if exists else '✗'
    print(f'  {status} {path.split("/content/")[1]}')
    if not exists:
        all_ok = False

print()
# Test import module
try:
    from src.finetuned.data.dataset_loader import DatasetLoader
    print('✓ Module import berhasil')
except ImportError as e:
    print(f'✗ Import error: {e}')
    print('  Pastikan src_finetuned.zip sudah di-extract dengan benar')
    all_ok = False

print()
print('✓ Setup selesai — siap lanjut!' if all_ok else '✗ Ada file yang belum tersedia, cek cell di atas')

Python path: ['/content', '/content', '/env/python']
Working dir: /content

=== File Check ===
  ✓ dataset_aqg/output_domain/train.jsonl
  ✓ dataset_aqg/output_domain/validation.jsonl
  ✓ dataset_aqg/output_domain/test.jsonl
  ✓ dataset_aqg/dataset-task-spesifc/train.jsonl
  ✓ dataset_aqg/dataset-task-spesifc/validation.jsonl
  ✓ dataset_aqg/dataset-task-spesifc/test.jsonl

✓ Module import berhasil

✓ Setup selesai — siap lanjut!


## 3. Check GPU

In [8]:
import torch

if not torch.cuda.is_available():
    print('⚠ GPU tidak tersedia!')
    print('Di Colab: Runtime → Change runtime type → T4 GPU')
    print('Di VS Code: Pastikan Colab kernel yang dipilih sudah enable GPU')
else:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {gpu_name}')
    print(f'✓ Memory: {gpu_memory:.1f} GB')
    
    if gpu_memory < 14:
        print(f'⚠ Memory {gpu_memory:.1f}GB mungkin kurang. Target: T4 (15GB)')

✓ GPU: Tesla T4
✓ Memory: 15.6 GB


## 4. Load dan Validate Datasets

In [9]:
from src.finetuned.data.dataset_loader import DatasetLoader

loader = DatasetLoader()

# Load domain dataset
print('Loading Domain Adaptation Dataset...')
domain_train = loader.load_dataset('/content/dataset_aqg/output_domain/', split='train')
domain_val   = loader.load_dataset('/content/dataset_aqg/output_domain/', split='validation')

print(f'Domain Train: {len(domain_train)} entries')
print(f'Domain Val:   {len(domain_val)} entries')

Loading Domain Adaptation Dataset...


Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 271 entries from /content/dataset_aqg/output_domain/train.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 33 entries from /content/dataset_aqg/output_domain/validation.jsonl
Domain Train: 271 entries
Domain Val:   33 entries


In [10]:
# Load task-specific dataset
print('Loading Task-Specific Dataset...')
task_train = loader.load_dataset('/content/dataset_aqg/dataset-task-spesifc/', split='train')
task_val   = loader.load_dataset('/content/dataset_aqg/dataset-task-spesifc/', split='validation')

print(f'Task Train: {len(task_train)} entries')
print(f'Task Val:   {len(task_val)} entries')

Loading Task-Specific Dataset...


Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 876 entries from /content/dataset_aqg/dataset-task-spesifc/train.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 175 entries from /content/dataset_aqg/dataset-task-spesifc/validation.jsonl
Task Train: 876 entries
Task Val:   175 entries


In [11]:
# Validate datasets
print('=== Domain Dataset Validation ===')
domain_val_result = loader.validate_dataset(domain_train)

print('\n=== Task-Specific Dataset Validation ===')
task_val_result = loader.validate_dataset(task_train)

# Preview sample
print('\n=== Sample Domain Entry ===')
print(f"Input:  {domain_train[0]['input'][:150]}...")
print(f"Target: {domain_train[0]['target'][:150]}...")

print('\n=== Sample Task Entry ===')
print(f"Input:  {task_train[0]['input'][:150]}...")
print(f"Target: {task_train[0]['target'][:150]}...")

=== Domain Dataset Validation ===

=== Dataset Validation Summary ===
Total Entries: 271
Duplicate Count: 18
Avg Input Length: 527.18 chars
Avg Target Length: 286.34 chars
Has Metadata: True
⚠ Warning: Found 18 duplicate entries

=== Task-Specific Dataset Validation ===

=== Dataset Validation Summary ===
Total Entries: 876
Duplicate Count: 1
Avg Input Length: 821.26 chars
Avg Target Length: 343.94 chars
Has Metadata: True
⚠ Warning: Found 1 duplicate entries

=== Sample Domain Entry ===
Input:  Apa itu Object (Objek) dalam Python?...
Target: | Nama | Deskripsi | Contoh |
|------|-----------|--------|
| **Class (Kelas)** | Cetakan (blueprint) untuk membuat objek-objek yang memiliki karakter...

=== Sample Task Entry ===
Input:  Konteks: ### Perbandingan Penggunaan Memori

```python
import numpy
import sys

var_list = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
var_array = numpy.array([...
Target: Pertanyaan: Sesuai catatan modul yang menggunakan list Python untuk matriks, lengkapi kode berikut u

## 5. Test Tokenizer

In [12]:
# IndoNanoT5 menggunakan AutoTokenizer (bukan T5Tokenizer)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('LazarusNLP/IndoNanoT5-base')
print(f'✓ Tokenizer loaded: {type(tokenizer).__name__}')
print(f'  Vocab size: {tokenizer.vocab_size}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


✓ Tokenizer loaded: T5Tokenizer
  Vocab size: 32000


In [13]:
from src.finetuned.data.tokenizer_tester import TokenizerTester

tester = TokenizerTester(tokenizer)

# Test markdown handling
print('Testing markdown handling...')
markdown_results = tester.test_markdown_handling()

# Analyze token distribution
print('\nAnalyzing domain dataset token distribution...')
domain_token_stats = loader.analyze_token_distribution(
    domain_train.select(range(min(50, len(domain_train)))),  # Sample 50 untuk speed
    tokenizer,
    max_length=512
)

print('\nAnalyzing task-specific dataset token distribution...')
task_token_stats = loader.analyze_token_distribution(
    task_train.select(range(min(50, len(task_train)))),
    tokenizer,
    max_length=512
)

Testing markdown handling...

=== Markdown Handling Test ===
✗ headings    : FAIL
  Original: # Heading 1
## Heading 2
  Decoded:  # eading 1 ## eading 2
✓ bold        : PASS
✓ code        : PASS
✗ newlines    : FAIL
  Original: line1
line2
line3
  Decoded:  line1 line2 line3
✗ mixed       : FAIL
  Original: # Heading
**Bold** dengan `code`
  Decoded:  # eading **old** dengan `code`

Analyzing domain dataset token distribution...

=== Token Distribution Analysis ===
Mean Length: 227.1 tokens
Median Length: 97 tokens
Max Length: 1592 tokens
Exceeding 512 tokens: 12.0%

Histogram:
       0-128:   31 ██████
     128-256:    4 
     256-384:    3 
     384-512:    6 █
     512-640:    2 
     640-768:    3 
    768-1024:    0 
       1024+:    1 

⚠ Warning: 12.0% of samples exceed max_length=512

Analyzing task-specific dataset token distribution...

=== Token Distribution Analysis ===
Mean Length: 438.6 tokens
Median Length: 390 tokens
Max Length: 909 tokens
Exceeding 512 tokens: 28.0%



## 6. Setup Model dengan LoRA

In [14]:
from src.finetuned.model.model_setup import ModelSetup
from peft import LoraConfig

model_setup = ModelSetup()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q', 'v'],
    bias='none',
    task_type='SEQ_2_SEQ_LM'
)

peft_model, tokenizer = model_setup.setup_model_for_training(
    model_name='LazarusNLP/IndoNanoT5-base',
    lora_config=lora_config
)


SETTING UP MODEL FOR TRAINING

Loading base model: LazarusNLP/IndoNanoT5-base


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

✓ Model loaded successfully
Loading tokenizer: LazarusNLP/IndoNanoT5-base
✓ Tokenizer loaded successfully
Applying LoRA adapters...
  Rank: 8
  Alpha: 16
  Dropout: 0.1
  Target modules: {'q', 'v'}
✓ LoRA adapters applied successfully

=== Model Parameter Summary ===
Trainable params: 884,736 (0.36%)
Total params:     248,462,592
Non-trainable:    247,577,856
✓ Parameter efficiency verified: 0.36% trainable

=== GPU Memory Status ===
Total Memory:     15.64 GB
Allocated Memory: 0.00 GB
Free Memory:      15.64 GB
✓ Sufficient GPU memory available

✓ Model setup complete!


In [15]:
# Test forward pass
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
peft_model = peft_model.to(device)

dummy_input = tokenizer('Test input Python', return_tensors='pt')
dummy_labels = tokenizer('Test output', return_tensors='pt')['input_ids']

dummy_input = {k: v.to(device) for k, v in dummy_input.items()}
dummy_labels = dummy_labels.to(device)

with torch.no_grad():
    outputs = peft_model(**dummy_input, labels=dummy_labels)

print(f'✓ Forward pass berhasil! Loss: {outputs.loss.item():.4f}')

✓ Forward pass berhasil! Loss: 10.1637


## 7. Baseline Evaluation

In [16]:
# Generate baseline predictions (10 samples)
print('Generating baseline predictions (10 samples)...')

samples = task_val.select(range(min(10, len(task_val))))
predictions = []
references  = []

for i, sample in enumerate(samples):
    inputs = tokenizer(
        sample['input'],
        return_tensors='pt',
        max_length=512,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        output_ids = peft_model.generate(**inputs, max_length=256, num_beams=2)
    
    prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    predictions.append(prediction)
    references.append(sample['target'])
    
    print(f'Sample {i+1}: {prediction[:80]}...')

# Compute BLEU
from evaluate import load
bleu = load('bleu')
baseline_bleu = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

print(f'\nBaseline BLEU-4: {baseline_bleu["bleu"]:.4f}')
print('Expected: < 0.15 (sebelum fine-tuning)')

Generating baseline predictions (10 samples)...
Sample 1: soal ini juga dapat digunakan untuk membuat kode blok yang dapat digunakan sebag...
Sample 2: `` ``` ### ### ### ### ### ### ### ### ### ### ### ### ### ### ###se..!`` ### ##...
Sample 3: --------------------------------------------------------------------------------...
Sample 4: `` ``` class class = class class class = `` class class class class class class ...
Sample 5: ---------------------------------------------------------------------------:----...
Sample 6: ` `=` untuk parameter `panjang`, kita perlu menambahkan nilai default untuk para...
Sample 7: , bahasa, bahasa, bahasa, bahasa, bahasa, bahasa, bahasa, bahasa, bahasa, bahasa...
Sample 8: ---- ## --- # ---- # ---- ## --- # --- # --- # --- --- --- ----: ----:---::: - -...
Sample 9: !!!!!!!!!!!!! - * - * - * - **jpeg: - **: --------------------------------: -...
Sample 10: --------------------------------------------------------------------------------...



Baseline BLEU-4: 0.0000
Expected: < 0.15 (sebelum fine-tuning)


## 8. Summary

In [17]:
print('\n' + '='*60)
print('VALIDATION SUMMARY')
print('='*60)

gpu_ok = torch.cuda.is_available()
print(f'\n1. GPU: {"✓ " + torch.cuda.get_device_name(0) if gpu_ok else "✗ Tidak tersedia"}')

print(f'\n2. Datasets:')
print(f'   Domain:        {len(domain_train)} train, {len(domain_val)} val')
print(f'   Task-Specific: {len(task_train)} train, {len(task_val)} val')

print(f'\n3. Token Distribution:')
print(f'   Domain exceed 512:        {domain_token_stats["pct_exceeding_limit"]:.1f}%')
print(f'   Task-Specific exceed 512: {task_token_stats["pct_exceeding_limit"]:.1f}%')

print(f'\n4. Model: ✓ IndoNanoT5-base + LoRA loaded')
print(f'\n5. Baseline BLEU-4: {baseline_bleu["bleu"]:.4f}')

print('\n' + '='*60)
if gpu_ok:
    print('✓ VALIDATION COMPLETE - Siap untuk training!')
    print('  Lanjut ke: 02_domain_adaptation.ipynb')
else:
    print('⚠ Enable GPU dulu sebelum training')
print('='*60)


VALIDATION SUMMARY

1. GPU: ✓ Tesla T4

2. Datasets:
   Domain:        271 train, 33 val
   Task-Specific: 876 train, 175 val

3. Token Distribution:
   Domain exceed 512:        12.0%
   Task-Specific exceed 512: 28.0%

4. Model: ✓ IndoNanoT5-base + LoRA loaded

5. Baseline BLEU-4: 0.0000

✓ VALIDATION COMPLETE - Siap untuk training!
  Lanjut ke: 02_domain_adaptation.ipynb
